In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [3]:
def quantum_random_bits(n):
    """Generate n random bits by measuring |+> = (|0>+|1>)/sqrt(2)"""
    bits = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)

        simulator = AerSimulator()
        compiled = transpile(qc, simulator)
        job = simulator.run(compiled, shots=1)
        result = job.result()
        counts = result.get_counts()
        bit = int(list(counts.keys())[0])
        bits.append(bit)
    return bits

# ALICE: Encode qubits
def alice_encode(bits, bases):
    """
    Alice encodes each bit into a qubit.
    bases: 0 = Z-basis (|0>, |1>), 1 = X-basis (|+>, |->)
    bits:  0 or 1
    """
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

# BOB: Measure qubits

def bob_measure(circuits, bases):
    """
    Bob measures each qubit in his randomly chosen basis.
    bases: 0 = Z-basis, 1 = X-basis
    """
    results = []
    simulator = AerSimulator()

    for qc, basis in zip(circuits, bases):
        measure_qc = qc.copy()
        if basis == 1:
            measure_qc.h(0)
        measure_qc.measure(0, 0)

        compiled = transpile(measure_qc, simulator)
        job = simulator.run(compiled, shots=1)
        result = job.result()
        counts = result.get_counts()
        bit = int(list(counts.keys())[0])
        results.append(bit)
    return results

# SIFTING

def sift_key(bits, alice_bases, bob_bases):
    """Keep bits where Alice and Bob used the same basis."""
    sifted = []
    matching_indices = []
    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            sifted.append(bits[i])
            matching_indices.append(i)
    return sifted, matching_indices

# ERROR CHECKING

def check_error_rate(alice_key, bob_key, sample_size=None):
    """
    Compare a sample of Alice's and Bob's sifted keys.
    Returns error rate. If > threshold → attack suspected.
    """
    if sample_size is None:
        sample_size = len(alice_key) // 4   # Use 25% as the check sample

    errors = 0
    for i in range(sample_size):
        if alice_key[i] != bob_key[i]:
            errors += 1

    return errors / sample_size if sample_size > 0 else 0


# MAIN PROTOCOL - NO ATTACKER
NUM_QUBITS = 100
THRESHOLD = 0.1

print("=" * 55)
print("       BB84 Protocol — No Attacker")
print("=" * 55)

# ALICE generates random bits and bases using quantum randomness
print("\n[ALICE] Generating random bits and bases...")
alice_bits  = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

# ALICE encodes her bits into qubits
print("[ALICE] Encoding qubits...")
encoded_qubits = alice_encode(alice_bits, alice_bases)

# BOB generates random measurement bases
print("[BOB]   Generating random measurement bases...")
bob_bases = quantum_random_bits(NUM_QUBITS)

# BOB measures the qubits
print("[BOB]   Measuring qubits...")
bob_results = bob_measure(encoded_qubits, bob_bases)

print("\n[PUBLIC] Alice and Bob compare bases publicly...")
alice_sifted, indices = sift_key(alice_bits, alice_bases, bob_bases)
bob_sifted, _         = sift_key(bob_results, alice_bases, bob_bases)

print(f"  Qubits sent:       {NUM_QUBITS}")
print(f"  Matching bases:    {len(alice_sifted)} ({len(alice_sifted)/NUM_QUBITS*100:.1f}%)")

# --- ERROR CHECKING ---
error_rate = check_error_rate(alice_sifted, bob_sifted)
print(f"\n[CHECK] Error rate on sample: {error_rate*100:.1f}%")

if error_rate > THRESHOLD:
    print("  ATTACK DETECTED! Error rate too high. Aborting.")
else:
    print("  Channel is clean. No attacker detected.")
    sample_size = len(alice_sifted) // 4
    final_key = alice_sifted[sample_size:]
    print(f"\n[KEY]   Shared secret key length: {len(final_key)} bits")
    print(f"[KEY]   First 10 bits: {final_key[:10]}")

    # Verify keys match
    keys_match = (alice_sifted[sample_size:] == bob_sifted[sample_size:])
    print(f"[CHECK] Alice's and Bob's keys match: {keys_match}")

       BB84 Protocol — No Attacker

[ALICE] Generating random bits and bases...
[ALICE] Encoding qubits...
[BOB]   Generating random measurement bases...
[BOB]   Measuring qubits...

[PUBLIC] Alice and Bob compare bases publicly...
  Qubits sent:       100
  Matching bases:    52 (52.0%)

[CHECK] Error rate on sample: 0.0%
  Channel is clean. No attacker detected.

[KEY]   Shared secret key length: 39 bits
[KEY]   First 10 bits: [1, 1, 1, 1, 0, 1, 0, 1, 1, 0]
[CHECK] Alice's and Bob's keys match: True
